# OMEGA feasibility — can SIMBA say anything about the z≈6–8 OMEGA galaxies?

A quick check before any Synthesizer nebular-line work: do SIMBA galaxies at z≈6–8 overlap the OMEGA parameter space
(M\*, SFR, T_e O/H, C/O, N/O), and are they resolved well enough (star particles, especially the ones younger than 10 Myr)?
Only quantities SIMBA already provides. The engine is in `omega_feasibility/*.py` (the Slurm job uses the same code);
this notebook drives it part by part.

- **Part 0** setup and knobs · **Part 1** build the catalogue (heavy: particles; or run `sbatch omega_feasibility/submit.sh`)
- **Part 2** resolution · **Part 3** overlap with OMEGA · **Part 4** merger trees (backward / forward)
- **Part 5** compact vs disky progenitors at z≈6–8 · **Part 6** the key figure · **Part 7** write the numbers

Snapshots (from the headers, identical in m100n1024 and m25n512): 026 z=7.963, 030 z=7.054, 036 z=5.930, 129 z=0.406.
Definitions: abundances are number ratios of the **star-forming gas** (SFR>0 particles) from SIMBA's tracked mass fractions,
n_X/n_H = (X_X/A_X)/(X_H/A_H) with X_H = 1 − Z − Y per particle, summed with SFR (or mass) weights; solar = Asplund+09
(12+log(O/H)⊙ = 8.69, log(C/O)⊙ = −0.26, log(N/O)⊙ = −0.86). SFR10/SFR100 from star-particle ages (current masses).
Compact / disky = median split of the stellar r80/r20 (caesar) among z≈0.4 quiescent centrals (log M\* > 10, sSFR < 0.2/t_H).

## Part 0 — setup and knobs
Run from anywhere inside the repo. Knobs are module attributes of `config`: change them here, the functions read them at call time.

In [ ]:
import os, sys
REPO = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(REPO, "omega_feasibility")) and REPO != "/":
    REPO = os.path.dirname(REPO)
os.chdir(REPO); sys.path.insert(0, REPO)          # the m25 tree sidecars are found under ./output/<sim>/progen_links

import importlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from omega_feasibility import config as C, build_catalogue as bc, analyse as an
for m in (C, bc, an):
    importlib.reload(m)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

# ── knobs ───────────────────────────────────────────────────────────────────
C.N_STAR_MIN    = 30      # star particles
C.N_YOUNG_MIN   = 5       # star particles younger than T_YOUNG_MYR
C.T_YOUNG_MYR   = 10.0
C.RES_FRACTION  = 0.8     # a mass bin is resolved when >= this fraction of its galaxies pass
C.KDE_FRACTION  = 0.95    # 'inside the SIMBA locus' = inside this KDE contour of the resolved analogues
C.OBS_CSV       = os.path.join(REPO, "omega_obs_z6-8.csv")
RUN_BUILD       = False   # True: build the catalogue here (Part 1, ~1 h for both boxes); False: read the products

print("repo:", REPO)
print("boxes:", C.SIMS, "| high-z snaps:", C.HIGHZ_SNAPS, "| low-z snap:", C.LOWZ_SNAP)
print("OMEGA table:", C.OBS_CSV, "(found)" if os.path.exists(C.OBS_CSV) else "(MISSING)")

## Part 1 — build the catalogue (heavy)

The only step that reads particles: per box, the z≈8/7/6 snapshots (members only: star ages and the star-forming gas
abundances), the z≈0.4 catalogue (quiescent sample + class) and the main branches 129 → 026 (catalogue columns).
Writes `output/<sim>/omega_feasibility/omega_{highz,lowz,progenitors}.parquet` (+ csv) and `omega_tracks.npz`.
With `RUN_BUILD = False` the cell only lists what already exists. Expect ~1 h and < 8 GB for both boxes (m100n1024 at z≈6 dominates);
the same build runs as one Slurm task with `sbatch omega_feasibility/submit.sh`.

In [ ]:
for sim in C.SIMS:
    d = C.out_dir(sim)
    have = {k: any(os.path.exists(os.path.join(d, f"omega_{k}{e}")) for e in (".parquet", ".csv", ".h5", ".npz"))
            for k in ("highz", "lowz", "progenitors", "tracks")}
    print(f"{sim:7s} ({C.SIMS[sim]}):", have)
    if not RUN_BUILD:
        continue
    s = bc.simulation(sim)
    df = pd.concat([bc.build_highz_snapshot(s, zt, sn) for zt, sn in C.HIGHZ_SNAPS.items()], ignore_index=True)
    bc.write_table(df, os.path.join(d, "omega_highz"))
    lowz = bc.build_lowz(s)
    bc.write_table(lowz, os.path.join(d, "omega_lowz"))
    prog, tracks = bc.build_trees(s, lowz)
    bc.write_table(prog, os.path.join(d, "omega_progenitors"))
    np.savez(os.path.join(d, "omega_tracks.npz"), **tracks)

## Part 2 — inputs, and the resolution of the z≈6–8 analogues
Analogues = star-forming centrals. The threshold is the lowest log M\* bin (0.25 dex) above which every populated bin has ≥ `RES_FRACTION` of its galaxies with ≥ `N_STAR_MIN` star particles (and ≥ `N_YOUNG_MIN` younger than 10 Myr). `sfr10_quantum` = the SFR10 one young particle is worth.

In [ ]:
R = an.compute()
obs = R["obs"]
display(obs[["id", "z", "zt", "logMstar", "logSFR", "OH12", "logCO", "logNO", "logNO_upper"]])
res = pd.DataFrame(R["res"])
display(res)

In [ ]:
an.fig_young(R["H"], R["res"], C.combined_dir()); plt.show()

## Part 3 — overlap with OMEGA
Each observed point is compared with the resolved analogues of the snapshot nearest its redshift. `inside` = within the 95 % KDE contour (an N/O upper limit counts as inside when any value below it is); `offset_y` = observed minus the SIMBA median at fixed x (±0.25 dex). SFR is Hα (OMEGA) against SFR10 (SIMBA). `above_mass_threshold` (M\*–O/H rows) = the point lies above that box's resolution mass.

In [ ]:
display(R["summ"])
display(R["ov"])

In [ ]:
an.fig_overlap(R["H"], R["obs"], C.combined_dir()); plt.show()

## Part 4 — merger trees
**Backward**: the main progenitors at z≈8/7/6 of the z≈0.4 quiescent compact / disky galaxies: how many exist, how many are resolved, and where they sit. **Forward**: of the resolved z≈6–8 analogues, the fraction on the main branch of a z≈0.4 galaxy, of a quiescent one and of each class (68 % Wilson intervals; a galaxy that merges into another branch counts as 'not').

In [ ]:
print("compact / disky split (median r80/r20 of the quiescent centrals):",
      {s: round(float(np.nanmedian(d["lowz"].loc[d["lowz"].quiescent.astype(bool), "C_r80_r20"])), 3) for s, d in R["tree_ok"].items()})
display(pd.DataFrame(R["brows"]))
display(pd.DataFrame(R["frows"]))

## Part 5 — do compact and disky progenitors already differ at z≈6–8?
Progenitors with ≥ `N_STAR_MIN` star particles. KS and Anderson–Darling (scipy caps the AD p-value to [0.001, 0.25]); the effect sizes are Cliff's δ (compact vs disky) and the median difference, with 68 % bootstrap intervals. `log_mstar` is listed as the confound to keep in mind.

In [ ]:
display(pd.DataFrame(R["trows"]))

## Part 6 — key figure: progenitor tracks in M\*–O/H
Median main-branch tracks of the compact and disky z≈0.4 quiescent galaxies (O/H along the track from caesar's SFR-weighted Z with the Asplund scaling; the markers at z≈8/7/6 are the true particle O/H of the same progenitors, the star is z≈0.4), the OMEGA points, and the resolution mass at each redshift (dotted).

In [ ]:
if R["back"]:
    an.fig_tracks({s: R["H"][s] for s in R["back"]}, R["D"], R["back"], R["res"], R["obs"], C.combined_dir()); plt.show()
else:
    print("no merger-tree products yet (Part 1)")

## Part 7 — write the numbers
`output/omega_feasibility/feasibility_numbers.json` + `summary_tables.md`: everything `FEASIBILITY.md` quotes.

In [ ]:
md = an.write_outputs(R)